# amazon26: Business Entity Resolution on Kaggle

This notebook runs the full pipeline and writes **`<TEAM_NAME>_submission.zip`** to `/kaggle/working`. Download it from the **Output** tab.

**Before you click Run All:**

1. **Add the challenge data** (*Add Input → Upload → New Dataset*). Upload the organisers' `student_resource/` folder, or just its `dataset/` folder. It must contain `train/train_source1.tsv` … `test/test_source3.tsv`. Keep the dataset **private**.
   If `utils/validate_submission.py` is included, the notebook runs it.
2. **Get the code**, in one of two ways:
   - *Settings → Internet → On*, and the notebook clones `REPO_URL`. For a private repo, add a GitHub token as a Kaggle secret named `GITHUB_TOKEN` (*Add-ons → Secrets*).
   - Or upload this repository as another Kaggle dataset. The notebook uses any input folder that contains `src/cli.py`.
3. Set `TEAM_NAME` in the next cell.

**Accelerator:** a GPU session (T4 or P100) is recommended. When the notebook sees a GPU, it trains the classifier with XGBoost on the GPU and fits it on a larger sample of Source 1 entities (`GPU_TRAIN_ENTITIES`). Normalisation, blocking and the pair features are sparse, string-heavy work that runs on the CPU cores in either kind of session. Without a GPU the notebook uses scikit-learn's gradient boosting on the CPU.

**Scale:** the pipeline is built for the full challenge data (about 2.2M Source 1 and 10M Source 2/3 records per split) on a standard 4-core / 30 GB CPU session. Blocking and the pair features run over all records; the classifier is fitted on a sample of `--max-train-entities` Source 1 entities (default 400,000). On a synthetic split a quarter of that size, train took 8 min at 7.2 GB peak and predict 5.5 min at 5.9 GB. At full size expect roughly 1.5–3 hours in total, most of it in blocking. Each stage logs its time and peak RAM. If memory runs out (exit code `-9`), lower `--max-train-entities` or `--max-df` in `EXTRA_ARGS`; if it is too slow, lower `--max-df` (fewer blocking keys, slightly lower recall).

In [ ]:
# ---- Configuration -------------------------------------------------------
TEAM_NAME = "amazon26"   # your registered team name -> <TEAM_NAME>_submission.zip
REPO_URL = "https://github.com/MounishSenisetty/amazon26.git"
REPO_BRANCH = "main"
EXTRA_ARGS = []          # extra flags for `src.cli train`, e.g. ["--max-train-entities", "200000", "--max-df", "2000"]
N_JOBS = 0               # worker processes for train and predict; 0 = all CPUs
USE_GPU = True           # if a GPU is visible: XGBoost on the GPU and a larger training sample
GPU_TRAIN_ENTITIES = 1_000_000   # Source 1 entities sampled for fitting when the GPU is used (CPU default: 400,000)
RUN_EVALUATE = True      # write validation_report.json (per-country scores) for docs/METHODOLOGY.md
METHODOLOGY_FILE = None  # path to a filled-in methodology .md; None = docs/METHODOLOGY.md from the repo

INPUT_ROOT = "/kaggle/input"
WORK_DIR = "/kaggle/working"
SCRATCH = "/tmp/amazon26"

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

INPUT_ROOT, WORK_DIR, SCRATCH = Path(INPUT_ROOT), Path(WORK_DIR), Path(SCRATCH)
OUT_DIR, MODEL_DIR = WORK_DIR / "output", WORK_DIR / "artifacts"
SCRATCH.mkdir(parents=True, exist_ok=True)


def sh(cmd, cwd=None, check=True, capture=False, env=None):
    """Run a command and stream its output into the notebook."""
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen([str(c) for c in cmd], cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    lines = []
    for line in proc.stdout:
        lines.append(line)
        if not capture:
            print(line, end="", flush=True)
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {' '.join(map(str, cmd))}")
    return proc.returncode, "".join(lines)


def find_first(pattern):
    hits = sorted(INPUT_ROOT.rglob(pattern)) if INPUT_ROOT.exists() else []
    return hits[0] if hits else None


mem_gb = int(next(l for l in open("/proc/meminfo") if l.startswith("MemTotal")).split()[1]) / 2**20
print(f"CPUs: {len(os.sched_getaffinity(0))} | RAM: {mem_gb:.0f} GB")

## 1. Locate the challenge data and the code

In [ ]:
first = find_first("train/train_source1.tsv")
if first is None:
    raise FileNotFoundError(f"No train/train_source1.tsv under {INPUT_ROOT}: add the challenge data as an input.")
DATA_DIR = first.parent.parent
missing = [f"{s}/{s}_source{n}.tsv" for s in ("train", "test") for n in (1, 2, 3)
           if not (DATA_DIR / s / f"{s}_source{n}.tsv").is_file()]
missing += [] if (DATA_DIR / "train" / "train_ground_truth.tsv").is_file() else ["train/train_ground_truth.tsv"]
if missing:
    raise FileNotFoundError(f"{DATA_DIR} is missing {missing}")
print("data:", DATA_DIR)

cli = find_first("src/cli.py")
if cli is not None:
    CODE_DIR = SCRATCH / "code"
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(cli.parent.parent, CODE_DIR, ignore=shutil.ignore_patterns("__pycache__", ".git", "dataset"))
    print("code: copied from input", cli.parent.parent)
else:
    url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = url.replace("https://", f"https://x-access-token:{token}@")
    except Exception:
        pass  # no secret: clone anonymously (public repo)
    CODE_DIR = SCRATCH / "repo"
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    rc, out = sh(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, url, CODE_DIR], check=False, capture=True)
    if rc != 0:
        raise RuntimeError("git clone failed. Turn Internet on, add a GITHUB_TOKEN secret for a private repo, "
                           "or upload the repo as a Kaggle dataset.\n" + out.replace(url, REPO_URL))
    print("code: cloned", REPO_URL, "@", REPO_BRANCH)
if not (CODE_DIR / "src" / "cli.py").is_file():
    raise FileNotFoundError(f"{CODE_DIR} has no src/cli.py; check REPO_BRANCH")

VALIDATOR = find_first("validate_submission.py")
print("organiser validator:", VALIDATOR or "not found (the pipeline's own rule checks will be used)")

## 2. Python environment

The notebook installs the exact versions in `requirements.txt` into a separate virtualenv (or, where Kaggle's Python has no `venv`, into a `--target` folder on `PYTHONPATH`), so the outputs match what reviewers reproduce from the zip. Without internet it falls back to Kaggle's preinstalled packages. The pipeline also works with pandas 2.x and scikit-learn ≥ 1.5, but the versions will differ from `requirements.txt`, so re-run with internet on before the final submission.

In [ ]:
PY, ENV = Path(sys.executable), dict(os.environ)
pip_flags = ["-q", "--disable-pip-version-check", "-r", CODE_DIR / "requirements.txt"]
VENV = SCRATCH / "venv"
if (sh([sys.executable, "-m", "venv", VENV], check=False)[0] == 0
        and sh([VENV / "bin" / "python", "-m", "pip", "install", *pip_flags], check=False)[0] == 0):
    PY = VENV / "bin" / "python"
else:
    print("\n!! venv unavailable; installing the pinned requirements into a --target folder instead.")
    SITE = SCRATCH / "site"
    shutil.rmtree(SITE, ignore_errors=True)
    if sh([sys.executable, "-m", "pip", "install", "--target", SITE, *pip_flags], check=False)[0] == 0:
        ENV["PYTHONPATH"] = str(SITE)
    else:
        print("\n!! Could not install pinned requirements; falling back to the preinstalled environment.")
        sh([PY, "-m", "pip", "install", "-q", "rapidfuzz"], check=False)
sh([PY, "-c", "import sys, pandas, numpy, sklearn, rapidfuzz; "
    "print('python', sys.version.split()[0], '| pandas', pandas.__version__, '| numpy', numpy.__version__, "
    "'| scikit-learn', sklearn.__version__, '| rapidfuzz', rapidfuzz.__version__)"], env=ENV)

## 3. Train, tune the threshold, predict on test

In [ ]:
gpu = USE_GPU and shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0
MODEL_ARGS = ["--model", "xgboost", "--device", "cuda", "--max-train-entities", GPU_TRAIN_ENTITIES] if gpu else []
print("GPU:", subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip() if gpu else "not used")

shutil.rmtree(OUT_DIR, ignore_errors=True)
# Separate processes, so all train-split memory is released before the test split loads.
# predict reuses the blocking settings that train saved in config.json.
sh([PY, "-m", "src.cli", "train", "--data-dir", DATA_DIR, "--model-dir", MODEL_DIR, "--n-jobs", N_JOBS, *MODEL_ARGS,
    *EXTRA_ARGS],
   cwd=CODE_DIR, env=ENV)
sh([PY, "-m", "src.cli", "predict", "--data-dir", DATA_DIR, "--model-dir", MODEL_DIR, "--out-dir", OUT_DIR,
    "--n-jobs", N_JOBS], cwd=CODE_DIR, env=ENV)

In [ ]:
if RUN_EVALUATE:
    # `train` already scored the held-out split (macro F0.5, blocking recall, per-country) and stored it in config.json.
    report = json.loads((MODEL_DIR / "config.json").read_text())["report"]
    (WORK_DIR / "validation_report.json").write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))

## 4. Validate the submission files

In [ ]:
if VALIDATOR:
    sh([PY, VALIDATOR, "--matching", OUT_DIR / "matching_results.tsv",
        "--candidate", OUT_DIR / "candidate_pairs.tsv", "--test-dir", DATA_DIR / "test"])
else:
    sh([PY, "-m", "src.cli", "check", "--data-dir", DATA_DIR, "--out-dir", OUT_DIR], cwd=CODE_DIR, env=ENV)

## 5. Build the submission zip

In [ ]:
cmd = [PY, CODE_DIR / "scripts" / "make_submission.py", "--team", TEAM_NAME, "--out-dir", OUT_DIR, "--dest", WORK_DIR]
if METHODOLOGY_FILE:
    cmd += ["--methodology", METHODOLOGY_FILE]
sh(cmd, env=ENV)
ZIP_PATH = WORK_DIR / f"{TEAM_NAME}_submission.zip"

try:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(ZIP_PATH, Path.cwd())))
except Exception:
    pass
print("Download from the Output tab:", ZIP_PATH)